T5モデルの実装をしてみる
あくまで、確認のため、データ数を1000として、trainデータセットのみ取ってくる

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from torch.utils.data import DataLoader

dataset = load_dataset("cnn_dailymail", "3.0.0", split="train[:1000]")
dataset = dataset.train_test_split(test_size=0.1)

print(dataset["train"][0])

I0000 00:00:1782103167.039209   13842 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782103167.580127   13842 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782103169.393624   13842 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

{'article': 'Japanese actress Rinko Kikuchi walks Anjali Rao through the streets of Tokyo. She stunned global cinema audiences with her controversial and Oscar-nominated performance as a lonely deaf girl in the film "Babel." Rinko Kikuchi is one of Japan\'s hottest young actresses and models, recently working with Karl Lagerfeld as the new face of Channel. Despite her success, she remains an unconventional figure in Japan, at odds with the traditional demure image of the Japanese woman and forging a career on her own terms. Talk Asia follows her on a modelling assignment, discusses how her life has changed since "Babel" and revisits the unique location of one of the film\'s most important scenes. E-mail to a friend .', 'highlights': 'Rinko Kikuchi was Oscar-nominated for her performance in the film "Babel"\nShe has recently worked with Karl Lagerfeld as the new face of Channel .\nShe challenges the traditional demure image of the Japanese woman .', 'id': 'd7783bd2bf5ad92156962380342411

In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 900
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 100
    })
})

モデルに関して
t5smallというモデルを用いる
語彙サイズや、最大トークンも確認
ちなみに、相対位置エンコーディングの場合、512トークン以上の長さは、モデル自体が破産するらしい

In [ ]:
model_checkpoint = "t5-small" 

# トークナイザとモデルのロード
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)


print(f"語彙サイズ (Vocab size): {model.config.vocab_size}") 
print(f"最大入力トークン数: {tokenizer.model_max_length}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

語彙サイズ (Vocab size): 32128
最大入力トークン数: 512


In [15]:
prefix = "summarize: "
max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    # 1. 入力文章の先頭にプレフィックスを付ける
    inputs = [prefix + doc for doc in examples["article"]]
    
    # 2. エンコーダ向けの入力をトークナイズ
    model_inputs = tokenizer(
        inputs, 
        max_length=max_input_length, 
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["highlights"], 
        max_length=max_target_length, 
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# データセット全体に前処理を一括適用 (バッチ処理で高速化)
tokenized_datasets = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [16]:
import numpy as np
import evaluate
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

valデータセットに対して、ルージュという指標を使う
stemmerで、言葉の形は一緒にしている(run,runningは同じ)

In [21]:
rouge = evaluate.load("rouge")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred#valデータに対する、回答、ラベルが返ってくる
    
    # モデルの予測IDをテキスト（文字列）にデコード
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    
    # 【重要】labelsの中にある「-100（Loss無視フラグ）」を、pad_token_idに戻す
    # （-100のままだとトークナイザがデコードできずにエラーになるため）
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGEスコアを計算
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    
    # 読みやすいように数値を100倍してパーセント表示にする
    return {k: round(v * 100, 4) for k, v in result.items()}

In [22]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-small-cnn-dailymail", # モデルの保存先
    eval_strategy="epoch",             # 評価のタイミング（1エポック終わるごとに評価）
    learning_rate=2e-5,                    # 学習率（T5ファインチューニングの標準的な値）
    per_device_train_batch_size=8,         # 訓練時のバッチサイズ
    per_device_eval_batch_size=8,          # 評価時のバッチサイズ
    weight_decay=0.01,                     # 過学習を防ぐための重み減衰
    save_total_limit=3,                    # 保存するチェックポイントの最大数
    num_train_epochs=3,                    # データセットを何周学習するか
    predict_with_generate=True,            # 【必須】評価時に実際にテキストを生成(デコード)させる
    fp16=False,                            # GPU(CUDA)環境があり、Tensorコアが使える場合はTrueにすると高速化・省メモリ
)

このdata_loaderは、

In [24]:
from transformers import DataCollatorForSeq2Seq


data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

In [25]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],   # 先ほど分割したtestデータを検証(Validation)として使用
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

/tmp/ipykernel_13842/1195077642.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


順調に値は下がっている

In [26]:
trainer.train()

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,1.993127,25.199400,10.745400,20.559300,20.551900
2,No log,1.942877,24.928600,10.698400,19.786600,19.798700
3,No log,1.935301,25.274900,11.103600,20.003200,20.023800


TrainOutput(global_step=339, training_loss=2.2409448314205385, metrics={'train_runtime': 36.2152, 'train_samples_per_second': 74.554, 'train_steps_per_second': 9.361, 'total_flos': 365422863974400.0, 'train_loss': 2.2409448314205385, 'epoch': 3.0})

３つの出力結果を出してみる

出力1は、ある程度正しく要約できているが、
2,3に関しては、ずれている
これは、学習データが少ないのが一番の原因であるが、このデータ量にしてはすごいと感じた

In [ ]:
import torch

model.eval()

num_samples = 3
sample_data = dataset["test"].select(range(num_samples))

for i, example in enumerate(sample_data):
    
    # 1. 入力テキストと正解の準備
    article = example["article"]
    reference = example["highlights"]
    input_text = "summarize: " + article
    inputs = tokenizer(
        input_text, 
        return_tensors="pt", # PyTorchのテンソル形式で出力
        max_length=512, 
        truncation=True
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad(): 
        outputs = model.generate(
            **inputs,
            max_length=128,          # 生成する要約の最大長
            num_beams=4,             # ★ビームサーチ：常に上位4つの候補を探索し、最も自然な文章を作る
            early_stopping=True,     # 終了合図(</s>)が出たら即ストップ
            no_repeat_ngram_size=3   # AI特有の「同じ3単語のフレーズを何度も繰り返す」バグを防止
        )
    
    # 4. 出力されたトークンIDを、人間が読める文字列にデコード
    prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    print(f"【元の記事 (Source - 先頭300文字)】\n{article[:300]}...\n")
    print(f"【人間の要約 (Reference)】\n{reference}\n")
    print(f"【AIの要約 (Prediction)】\n{prediction}\n")
    print("=" * 60 + "\n")

【元の記事 (Source - 先頭300文字)】
ATLANTA, Georgia (CNN) -- It's midnight on the streets of Atlanta, and bar owner Rufus Terrill patrols his neighborhood with a rolling crime fighter of his own creation. Meet "Bum-bot," as Terrill describes it; others in his neighborhood call it simply, "Robocop." This former BBQ smoker is armed wit...

【人間の要約 (Reference)】
Atlanta bar owner converts BBQ smoker into robot to chase off drug dealers, others .
"Bum-bot" is armed with a water gun and loudspeaker .
Cops frown upon spraying water at bystanders, say it could be assault .

【AIの要約 (Prediction)】
"Bum-bot" is a barbecue smoker mounted on a three-wheeled scooter. It's armed with a water gun to chase off bums and drug dealers in downtown Atlanta. More than 20 suspicious people were seen huddling in the dark on this night.


【元の記事 (Source - 先頭300文字)】
(CNN) -- China's economy is booming and Sheikh Mohammed Bin Rashid Al Maktoum's visit there this week highlights the U.A.E.'s ambitions to join in on this growt

In [33]:
trainer.save_model("./t5-small-cnn-final")
tokenizer.save_pretrained("./t5-small-cnn-final")

('./t5-small-cnn-final/tokenizer_config.json',
 './t5-small-cnn-final/special_tokens_map.json',
 './t5-small-cnn-final/spiece.model',
 './t5-small-cnn-final/added_tokens.json',
 './t5-small-cnn-final/tokenizer.json')